In [2]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Is GPU available? True
GPU Name: Tesla T4


In [1]:
import torch
import tensorflow as tf
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Load the MNIST dataset (using keras for convenience as it returns numpy arrays)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

print(f"{x_train[0].shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
x_train shape: (60000, 28, 28)
y_train shape: (60000,)
x_test shape: (10000, 28, 28)
y_test shape: (10000,)
(28, 28)


Next, we need to preprocess the data. This involves flattening the images from 28x28 pixels to a 784-dimensional vector and normalizing the pixel values to be between 0 and 1. We also need to one-hot encode the labels.

In [ ]:
# Preprocess the data
# Flatten the images from (28, 28) to (784,) and normalize pixel values to [0, 1]
x_train = torch.tensor(x_train.reshape(-1, 784)).float() / 255.0
x_test = torch.tensor(x_test.reshape(-1, 784)).float() / 255.0

# Convert labels to long tensors (PyTorch CrossEntropyLoss expects class indices)
y_train = torch.tensor(y_train).long()
y_test = torch.tensor(y_test).long()

# Create TensorDatasets and DataLoaders
batch_size = 64

train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"x_train shape after preprocessing: {x_train.shape}")
print(f"y_train shape after preprocessing: {y_train.shape}")

x_train shape after preprocessing: (60000, 784)
y_train shape after preprocessing: (60000, 10)
(10,)


In [ ]:
print(f"First training label: {y_train[0]}")

[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


Now, let's define our Multi-Layer Perceptron (MLP) model using Keras's Sequential API. This model will consist of several dense layers with ReLU activation and a final output layer with softmax activation for classification.

In [ ]:
# Define the MLP model using PyTorch's nn.Module
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MLP, self).__init__()
        self.layer1 = nn.Linear(input_size, 256)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.layer3(x)
        return x

input_size = 784
num_classes = 10
model = MLP(input_size, num_classes)

print(model)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

Next, we need to compile the model. We'll use the Adam optimizer, categorical crossentropy loss (because our labels are one-hot encoded), and accuracy as our metric.

In [ ]:
# Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss() # PyTorch CrossEntropyLoss expects class indices, not one-hot encoded labels
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model, Loss function, and Optimizer defined successfully!")

Model compiled successfully!


Finally, let's train the model using the preprocessed training data and evaluate its performance on the test data.

In [ ]:
# Train the model
epochs = 10

for epoch in range(epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (inputs, labels) in enumerate(train_loader):
        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_samples
    print(f'Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}')

print("\nEvaluating the model on test data...")
model.eval() # Set model to evaluation mode
test_loss = 0.0
correct_predictions = 0
total_samples = 0

with torch.no_grad(): # Disable gradient calculation for evaluation
    for inputs, labels in test_loader:
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

final_test_loss = test_loss / len(test_loader)
final_test_accuracy = correct_predictions / total_samples

print(f"Test Loss: {final_test_loss:.4f}")
print(f"Test Accuracy: {final_test_accuracy:.4f}")

Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - categorical_accuracy: 0.9257 - loss: 0.2496 - val_categorical_accuracy: 0.9697 - val_loss: 0.1027
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - categorical_accuracy: 0.9705 - loss: 0.0960 - val_categorical_accuracy: 0.9737 - val_loss: 0.0849
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - categorical_accuracy: 0.9803 - loss: 0.0635 - val_categorical_accuracy: 0.9802 - val_loss: 0.0669
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - categorical_accuracy: 0.9860 - loss: 0.0457 - val_categorical_accuracy: 0.9762 - val_loss: 0.0796
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - categorical_accuracy: 0.9888 - loss: 0.0343 - val_categorical_accuracy: 0.9793 - val_loss: 0.0723
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - categorical_accuracy: 0.9910 - loss: 0.0288 - val_categorical_accuracy: 0.9782 - val_loss: 0.0803
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - categorical_accuracy: 0.9917 - los